<a href="https://colab.research.google.com/github/noname123920/diplom/blob/main/PINN2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
import numpy as np
def relative_l2_error(model, exact_fn, n_x=200, n_t=100,
                      domain_x=(0, np.pi), domain_t=(0, 1)):
    x = np.linspace(domain_x[0], domain_x[1], n_x)
    t = np.linspace(domain_t[0], domain_t[1], n_t)
    X, T = np.meshgrid(x, t)
    inputs = np.stack([X.ravel(), T.ravel()], axis=1).astype(np.float32) # расчетная сетка из 20000 точек (200x100)
    p_pinn  = model(inputs).numpy().reshape(X.shape)
    p_exact = exact_fn(X, T)
    return np.linalg.norm(p_pinn - p_exact) / np.linalg.norm(p_exact)

## 1.1. Piezoconductivity Equation (Dirichlet)

Solve the 1D piezoconductivity equation: $p_t = \chi p_{xx}$ for $x$ in $[0, \pi]$, $t$ in $[0, 1]$, with initial condition $p(x, 0) = \sin(x)$, boundary conditions $p(0,t) = 0$, $p(\pi,t) = 0$.

In [13]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

# Parameters
chi = 1
domain_x = [0, np.pi]
domain_t = [0, 1]

# Neural Network
class PINN(tf.keras.Model):
  def __init__(self):
    super(PINN, self).__init__()
    self.dense1 = tf.keras.layers.Dense(20, activation='tanh')
    self.dense2 = tf.keras.layers.Dense(20, activation='tanh')
    self.dense3 = tf.keras.layers.Dense(1)

  def call(self, inputs):
    x = inputs[:, 0:1]
    t = inputs[:, 1:2]

    u = self.dense1(inputs)
    u = self.dense2(u)
    u = self.dense3(u)

    return tf.sin(x) + x * (np.pi - x) * t * u

# PDE residual
def pde_residual(model, x, t):
  with tf.GradientTape(persistent=True) as tape:
    tape.watch(x)
    tape.watch(t)
    inputs = tf.concat([x, t], axis=1)
    p = model(inputs)
    p_t = tape.gradient(p, t)
    p_x = tape.gradient(p, x)
  p_xx = tape.gradient(p_x, x)
  del tape
  return p_t - chi * p_xx

# Training
model = PINN()
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)
loss_history_dirichlet = []
for epoch in range(1000):
  x = tf.random.uniform((100, 1), domain_x[0], domain_x[1])
  t = tf.random.uniform((100, 1), domain_t[0], domain_t[1])
  with tf.GradientTape() as tape:
    residual = pde_residual(model, x, t)
    loss_pde = tf.reduce_mean(tf.square(residual))
    loss_dirichlet = loss_pde
  grads = tape.gradient(loss_dirichlet, model.trainable_variables)
  optimizer.apply_gradients(zip(grads, model.trainable_variables))
  if epoch % 50 == 0:
    loss_val = float(loss_dirichlet.numpy())
    loss_history_dirichlet.append(loss_val)
    print(f"Epoch {epoch}, Loss: {loss_dirichlet.numpy()}")

err_dirichlet = relative_l2_error(model, lambda x, t: np.exp(-t) * np.sin(x))
print(f"Dirichlet: relative L2 error = {err_dirichlet:.4e}")

Epoch 0, Loss: 3.0331482887268066
Epoch 50, Loss: 0.05266036093235016
Epoch 100, Loss: 0.022786810994148254
Epoch 150, Loss: 0.009816087782382965
Epoch 200, Loss: 0.006086025387048721
Epoch 250, Loss: 0.004142298828810453
Epoch 300, Loss: 0.002321468200534582
Epoch 350, Loss: 0.0022381646558642387
Epoch 400, Loss: 0.002563049551099539
Epoch 450, Loss: 0.0020327202510088682
Epoch 500, Loss: 0.0021425499580800533
Epoch 550, Loss: 0.0020015970803797245
Epoch 600, Loss: 0.002285668393597007
Epoch 650, Loss: 0.0014747829409316182
Epoch 700, Loss: 0.0016875537112355232
Epoch 750, Loss: 0.0015150298131629825
Epoch 800, Loss: 0.0013014046708121896
Epoch 850, Loss: 0.001347729004919529
Epoch 900, Loss: 0.0009199075284413993
Epoch 950, Loss: 0.0011288204696029425
Dirichlet: relative L2 error = 8.1319e-03


In [14]:
import plotly.express as px
import pandas as pd

time_steps = np.linspace(domain_t[0], domain_t[1], 51)
plot_data = []

for t_val in time_steps:
  x_plot = np.linspace(domain_x[0], domain_x[1], 100)[:, None]
  t_plot = t_val * np.ones_like(x_plot)
  inputs_plot = np.concatenate([x_plot, t_plot], axis=1)
  p_plot = model(inputs_plot).numpy()
  p_exact = np.exp(-t_val) * np.sin(x_plot)
  df_temp = pd.DataFrame({
      'x': x_plot.flatten(),
      'PINN': p_plot.flatten(),
      'Exact': p_exact.flatten(),
      'Time': t_val
  })
  plot_data.append(df_temp)

df_plot = pd.concat(plot_data)
df_melted = df_plot.melt(id_vars=['x', 'Time'], var_name='Solution Type', value_name='p')
fig = px.line(df_melted, x='x', y='p', animation_frame='Time', color='Solution Type',
              title='1D Piezoconductivity Equation Solution Over Time')
fig.update_layout(
    xaxis_title='x',
    yaxis_title='p(x, t)'
)
fig.show()

## LOSS:

In [15]:
import plotly.graph_objects as go

epochs = [0, 50, 100, 150, 200, 250, 300, 350, 400, 450, 500,
          550, 600, 650, 700, 750, 800, 850, 900, 950]

fig = go.Figure()
fig.add_trace(go.Scatter(x=epochs, y=loss_history_dirichlet, mode='lines+markers', name='Дирихле'))
fig.update_layout(
    title='Сходимость обучения PINN',
    xaxis_title='Эпоха',
    yaxis_title='Функция потерь',
    yaxis_type='log',  # логарифмическая шкала
    template='plotly_white'
)
fig.show()